# ASR Pretraining


In [2]:
# install required packages
!pip install evaluate
!pip3 install torch torchvision torchaudio datasets transformers soundfile jiwer --index-url https://download.pytorch.org/whl/cu118
!pip3 install librosa --index-url https://pypi.org/simple

Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement jiwer (from versions: none)
ERROR: No matching distribution found for jiwer


In [3]:
import os

import re
import torch
import torch.nn as nn
import numpy as np

from datasets import load_dataset, disable_caching
import evaluate
from transformers import Wav2Vec2ForPreTraining, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Encoder


## Fine-tuning Wav2Vec2 with CTC Loss (5 points)

Build a fine-tuning pipeline for a pretrained multilingual Wav2Vec2 model on Belarusian audio from the [Fleurs](https://huggingface.co/datasets/google/fleurs) dataset.


#### Data Preparation


In [4]:
train = load_dataset("google/fleurs", "ru_ru", split="train")
test = load_dataset("google/fleurs", "ru_ru", split="test")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
print(train)
print(test)

Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 2562
})
Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 775
})


Preprocessing steps:

* Filter samples where `transcription` contains digits. Note: account for Belarusian-specific characters "і" and "ў"
* Strip punctuation from `transcription`


In [6]:
def filter_on_simb(data):
    text = data["transcription"]
    if text is None:
        return False

    text = text.lower()

    text = re.sub(r"[—–−]", "-", text)

    allowed_pattern = re.compile(r"^[а-яё\s\-]+$")

    return bool(allowed_pattern.match(text))

preprocessed_train = train.filter(filter_on_simb)
preprocessed_val = test.filter(filter_on_simb)

#### Tokenizer Training


Train a BPE tokenizer on Fleurs transcriptions using the [HuggingFace tokenizers library](https://huggingface.co/docs/tokenizers/en/training_from_memory).


In [7]:
import torch

class UzbekTokenizer:
    def __init__(self):
        # Index 0 must be your CTC blank/pad token
        self.blank_token = "_"

        self.chars = [
            self.blank_token, ' ', 'а', 'б', 'в', 'г', 'д', 'е', 'ё', 'ж', 'з', 'и', 'й',
            'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ',
            'ъ', 'ы', 'ь', 'э', 'ю', 'я'
        ]
        self.char_to_id = {char: idx for idx, char in enumerate(self.chars)}
        self.id_to_char = {idx: char for idx, char in enumerate(self.chars)}

    def get_vocab_size(self):
        return len(self.chars)

    def token_to_id(self, token):
        return self.char_to_id.get(token, 0)

    def encode(self, text):
        text = text.lower().strip()
        tokens = []
        i = 0
        while i < len(text):
            if i + 2 <= len(text) and text[i:i+2] in self.char_to_id:
                tokens.append(self.char_to_id[text[i:i+2]])
                i += 2
            elif text[i] in self.char_to_id:
                tokens.append(self.char_to_id[text[i]])
                i += 1
            else:
                i += 1
        return tokens

    def decode(self, token_ids):
        decoded_chars = []
        prev_token = None

        for token in token_ids:
            token = token.item() if isinstance(token, torch.Tensor) else token


            if token == prev_token:
                continue

            if token != 0:
                decoded_chars.append(self.id_to_char.get(token, ""))

            prev_token = token

        return "".join(decoded_chars).strip()

    @property
    def pad_token_id(self):
        return 0

    @property
    def vocab_size(self):
        return len(self.chars)

In [8]:
from tokenizers import Tokenizer, models, trainers, tokenizers, normalizers, pre_tokenizers, decoders

PAD_TOKEN = "[PAD]"
BOS_TOKEN = "[BOS]"
EOS_TOKEN = "[EOS]"
UNK_TOKEN = "[UNK]"
VOCAB_SIZE = 1000

tokenizer = UzbekTokenizer()

#### Model and Feature Extractor


In [9]:
from transformers import Wav2Vec2FeatureExtractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
   "facebook/wav2vec2-xls-r-300m"
)
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.token_to_id(PAD_TOKEN),
    vocab_size=tokenizer.get_vocab_size(),
)


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Data Processor and Collator


In [10]:
class CtcDataProcessor:
    def __init__(self, tokenizer, feature_extractor):
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor # probably mel spectogram

    def __call__(self, row):
        """
            Function applies tokenizer on row['transcription'] and applies feature extractor on audio column in row.
            Input: dict with transcription and audio fields
            Output: original dict includes `labels` column with tokenized sequence and `input_values` column with computed spectrogram.
        """

        labels = self.tokenizer.encode(row['transcription'])
        audio_array = row["audio"]["array"]
        sampling_rate = row["audio"]["sampling_rate"]

        features = self.feature_extractor(
            audio_array,
            sampling_rate=sampling_rate,
            return_tensors="np"
        )

        return {
            "input_values": features.input_values[0].tolist(),
            "labels": labels
        }



In [11]:
data_processor = CtcDataProcessor(tokenizer, feature_extractor)
train = preprocessed_train.map(data_processor, keep_in_memory=False, remove_columns=preprocessed_train.column_names)
val = preprocessed_val.map(data_processor, keep_in_memory=False, remove_columns=preprocessed_val.column_names)

Map:   0%|          | 0/1857 [00:00<?, ? examples/s]

Map:   0%|          | 0/568 [00:00<?, ? examples/s]

In [12]:

class CTCDataCollator:
    LABELS_PAD_IDX = -100

    @staticmethod
    def collate_tokens(tokens_batch, type, pad_value=0.0):
        max_len = max(len(seq) for seq in tokens_batch)

        padded_seqs = []
        for seq in tokens_batch:
            pad_len = max_len - len(seq)

            if type == "audio":
                padded = seq + [float(pad_value)] * pad_len
                padded_seqs.append(torch.tensor(padded, dtype=torch.float32))
            else:
                padded = seq + [int(pad_value)] * pad_len
                padded_seqs.append(torch.tensor(padded, dtype=torch.long))

        return torch.stack(padded_seqs)

    def __call__(self, batch):
        """
        Function collates `input_values` and `labels` into unified tensors.
        Input: list of dicts (output of CTCDataProcessor)
        Output: dict containing batched 'input_values', 'labels', and 'attention_mask'
        """
        input_values = [item["input_values"] for item in batch]
        labels = [item["labels"] for item in batch]

        batched_inputs = self.collate_tokens(input_values, type="audio", pad_value=0.0)

        batched_labels = self.collate_tokens(labels, type="labels", pad_value=self.LABELS_PAD_IDX)

        attention_mask = torch.zeros(batched_inputs.shape, dtype=torch.long)
        for i, seq in enumerate(input_values):
            attention_mask[i, :len(seq)] = 1

        return {
            "input_values": batched_inputs,
            "labels": batched_labels,
            "attention_mask": attention_mask
        }

#### Inference and Metrics

Use a greedy strategy to decode CTC output.

Note: remember that padding uses -100, and CTC output requires blank collapsing.


In [14]:
!pip install jiwer
from itertools import groupby
wer_metric = evaluate.load("wer")

class MetricsComputer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, pred):
        preds_logits = pred.predictions
        label_ids = pred.label_ids

        pred_ids = np.argmax(preds_logits, axis=-1)
        print(f"Raw model index predictions for first sample: {pred_ids[0][:20]}")

        pred_str = []
        label_str = []

        for i in range(len(pred_ids)):
            p_str = self.tokenizer.decode(pred_ids[i])
            pred_str.append(p_str)

            clean_label_ids = [token for token in label_ids[i] if token != -100]

            l_str = self.tokenizer.decode(clean_label_ids)
            label_str.append(l_str)

        if len(pred_str) > 0:
            print(f"\n--- Validation Sample ---")
            print(f"Prediction: {pred_str[0]}")
            print(f"Reference:  {label_str[0]}\n-------------------------")

        wer = wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": wer}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.0 MB/s eta 0:00:00


#### Training Sanity Check

Verify the pipeline by overfitting on a small batch first. Then fine-tune to reach WER ≤ 50 on the validation set.


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="test",
    per_device_train_batch_size=2, # you could increase batch size
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    max_steps=3000,
    fp16=torch.cuda.is_available(),
    save_steps=200,
    eval_steps=200,
    logging_steps=10,
    learning_rate=3e-4,#base (can be lower for such big model)
    weight_decay=0.1, # base too
    warmup_steps=300, #10% base
)

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=CTCDataCollator(),
    args=training_args,
    compute_metrics=MetricsComputer(tokenizer=tokenizer),
    train_dataset=train,
    eval_dataset=val,
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
200,24.902963,3.136078,1.000000
400,20.316679,2.368061,1.000000
600,8.661048,0.872365,0.931031
800,5.961106,0.591731,0.752886
1000,4.389603,0.493968,0.638127
1200,2.932915,0.439407,0.578619
1400,2.223021,0.398741,0.522138
1600,1.992367,0.404907,0.494134
1800,1.584917,0.399269,0.459224
2000,1.360028,0.423418,0.445601


Raw model index predictions for first sample: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- Validation Sample ---
Prediction: 
Reference:  в древнем китае использовали уникальный способ обозначения периодов времени каждый этап китая или каждая семья находившаяся у власти были особой династией
-------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Raw model index predictions for first sample: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- Validation Sample ---
Prediction: рттстонкстспснчпторктттктсткпчстстпст
Reference:  в древнем китае использовали уникальный способ обозначения периодов времени каждый этап китая или каждая семья находившаяся у власти были особой династией
-------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Raw model index predictions for first sample: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- Validation Sample ---
Prediction: в древнем китай использовали уникальный спосоп обозначение передов в времени каждый этоп китая или каждая с семья находившееся у власти были особой династийй
Reference:  в древнем китае использовали уникальный способ обозначения периодов времени каждый этап китая или каждая семья находившаяся у власти были особой династией
-------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Raw model index predictions for first sample: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- Validation Sample ---
Prediction: в древнием китаи использовали уникальный спосоп обозначение передов в времени каждый этап китая или каждая с семья находившееся у власти были особой династий
Reference:  в древнем китае использовали уникальный способ обозначения периодов времени каждый этап китая или каждая семья находившаяся у власти были особой династией
-------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]